# Statistical Analysis of Italian Real Estate Market Dynamics

This notebook moves from descriptive segmentation to **statistical analysis**. The objective is to test whether demographic growth and transaction activity are statistically associated with subsequent OMI quotation growth at municipality level.

**Analytical pipeline:** raw OMI + NTN + population → annual municipality panel → lagged variables → descriptive statistics → Pearson/Spearman correlations → OLS regression → diagnostics → robustness checks.

> **Important:** the models are exploratory and associative, not causal. OMI quotations are market quotation ranges rather than observed transaction prices; NTN measures transaction volume; omitted variables, simultaneity and measurement differences can bias estimated relationships.

## 1. Setup and analytical design

We use the **same-semester annual lag** rather than a simple row-wise `pct_change()`. This avoids treating the duplicated annual population value in S1/S2 as a real six-month demographic change.

Main specification:

`price_growth_t = β0 + β1 population_growth_(t-1) + β2 NTN_growth_(t-1) + region effects + year effects + ε`

The lag structure is deliberately conservative: explanatory variables are measured in the previous year, while the dependent variable is current price growth.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

QUOTATIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'quotations'
TRANSACTIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'transactions'
POPULATION_DIR = PROJECT_ROOT / 'data' / 'raw' / 'population'
for path in [QUOTATIONS_DIR, TRANSACTIONS_DIR, POPULATION_DIR]:
    assert path.exists(), f'Missing source directory: {path}'

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 2. Rebuild the municipality-semester panel

The notebook is independently executable and does not depend on variables created by previous notebooks.

In [ ]:
quotation_parts = []
for path in sorted(QUOTATIONS_DIR.glob('omi_quotations_*.csv')):
    match = re.search(r'_(\d{4})_(S[12])$', path.stem)
    if not match:
        continue
    year, semester = int(match.group(1)), match.group(2)
    df = pd.read_csv(path, sep=';', low_memory=False)
    df.columns = [str(c).replace('\ufeff', '').strip() for c in df.columns]
    required = ['Comune_ISTAT', 'Descr_Tipologia', 'Compr_min', 'Compr_max', 'Regione']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'{path.name}: missing columns {missing}')
    df = df[required].copy()
    df['year'], df['semester'] = year, semester
    quotation_parts.append(df)

omi = pd.concat(quotation_parts, ignore_index=True)
omi['municipality_code'] = omi['Comune_ISTAT'].astype('string').str.strip()
omi['Compr_min'] = pd.to_numeric(omi['Compr_min'], errors='coerce')
omi['Compr_max'] = pd.to_numeric(omi['Compr_max'], errors='coerce')
omi['price_m2'] = omi[['Compr_min', 'Compr_max']].mean(axis=1)
residential = omi[omi['Descr_Tipologia'].astype('string').str.contains('abitazion|villa', case=False, na=False)].copy()
price_panel = (residential.dropna(subset=['municipality_code', 'price_m2'])
    .groupby(['year', 'semester', 'municipality_code'], as_index=False)
    .agg(price_m2=('price_m2', 'median'), quotation_obs=('price_m2', 'size'), region=('Regione', 'first')))

transaction_parts = []
for folder in sorted(p for p in TRANSACTIONS_DIR.iterdir() if p.is_dir() and p.name.isdigit()):
    year = int(folder.name)
    files = [p for p in folder.iterdir() if p.is_file()]
    lista_path = next(p for p in files if 'lista-com' in p.name.lower())
    res_path = next(p for p in files if 'valori-res' in p.name.lower())
    lista = pd.read_csv(lista_path, sep=';', decimal=',')
    res = pd.read_csv(res_path, sep=';', decimal=',')
    lista.columns = [str(c).strip() for c in lista.columns]
    res.columns = [str(c).strip() for c in res.columns]
    lista_code = next(c for c in lista.columns if re.search(r'codcom$', c, re.I))
    res_code = next(c for c in res.columns if re.search(r'codcom$', c, re.I))
    ntn_candidates = [c for c in res.columns if re.fullmatch(r'NTN_?'+str(year), c, re.I)]
    if not ntn_candidates:
        ntn_candidates = [c for c in res.columns if re.match(r'NTN', c, re.I) and 'mq' not in c.lower()]
    ntn_col = ntn_candidates[0]
    geo_cols = [c for c in [lista_code, 'Comune', 'Provincia', 'Regione'] if c in lista.columns]
    geo = lista[geo_cols].rename(columns={lista_code: 'municipality_code'}).copy()
    vals = res[[res_code, ntn_col]].rename(columns={res_code: 'municipality_code', ntn_col: 'ntn'}).copy()
    vals['ntn'] = pd.to_numeric(vals['ntn'], errors='coerce')
    if vals['municipality_code'].duplicated().any():
        raise ValueError(f'{year}: duplicate NTN municipality key')
    transaction_parts.append(geo.merge(vals, on='municipality_code', how='left', validate='one_to_one').assign(year=year))
transactions = pd.concat(transaction_parts, ignore_index=True)

population_parts = []
for folder in sorted(p for p in POPULATION_DIR.iterdir() if p.is_dir() and p.name.isdigit()):
    year = int(folder.name)
    path = next(folder.glob('*_Comuni.csv'))
    df = pd.read_csv(path, sep=';', encoding='utf-8-sig', usecols=['Codice comune', 'Età', 'Totale'])
    df.columns = ['municipality_code', 'age', 'population']
    df['municipality_code'] = df['municipality_code'].astype('string').str.strip()
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['population'] = pd.to_numeric(df['population'], errors='coerce')
    totals = df.loc[df['age'].eq(999), ['municipality_code', 'population']].copy()
    totals['year'] = year
    population_parts.append(totals)
population = pd.concat(population_parts, ignore_index=True)

market = price_panel.merge(transactions[['year', 'municipality_code', 'ntn']], on=['year', 'municipality_code'], how='left', validate='many_to_one')
market = market.merge(population, on=['year', 'municipality_code'], how='left', validate='many_to_one')
market['period'] = market['year'].astype(str) + '-' + market['semester']
assert not market.duplicated(['year', 'semester', 'municipality_code']).any()
print(f'Panel rows: {len(market):,}')
display(market.head())

## 3. Annual municipality panel and explicit lags

For each municipality-year, the dependent variable is the **S2-to-S2 price growth**. Population and NTN growth are measured over the same annual interval and then lagged by one year in the regression. This prevents S1/S2 duplication of annual variables.

In [ ]:
annual_price = (market.assign(semester_num=market['semester'].str[-1].astype(int))
    .sort_values(['municipality_code', 'year', 'semester_num'])
    .groupby(['municipality_code', 'year'], as_index=False)
    .agg(price_m2=('price_m2', 'mean'), region=('region', 'first'), population=('population', 'first'), ntn=('ntn', 'first'),
         quotation_obs=('quotation_obs', 'sum')))

annual = annual_price.sort_values(['municipality_code', 'year']).copy()
annual['price_growth_pct'] = annual.groupby('municipality_code')['price_m2'].pct_change() * 100
annual['population_growth_pct'] = annual.groupby('municipality_code')['population'].pct_change() * 100
annual['ntn_growth_pct'] = annual.groupby('municipality_code')['ntn'].pct_change() * 100
annual['lag_population_growth_pct'] = annual.groupby('municipality_code')['population_growth_pct'].shift(1)
annual['lag_ntn_growth_pct'] = annual.groupby('municipality_code')['ntn_growth_pct'].shift(1)
annual['lag_price_growth_pct'] = annual.groupby('municipality_code')['price_growth_pct'].shift(1)

analysis = annual.replace([np.inf, -np.inf], np.nan).copy()
print(f'Annual rows: {len(analysis):,}')
display(analysis[['year', 'municipality_code', 'price_m2', 'ntn', 'population', 'price_growth_pct', 'lag_population_growth_pct', 'lag_ntn_growth_pct']].head(10))

## 4. Data quality and descriptive statistics

Before estimating relationships, inspect missingness, sample size and the distribution of the variables entering the model.

In [ ]:
model_cols = ['price_growth_pct', 'lag_population_growth_pct', 'lag_ntn_growth_pct', 'lag_price_growth_pct']
quality = pd.DataFrame({
    'non_null': analysis[model_cols].notna().sum(),
    'missing_pct': analysis[model_cols].isna().mean() * 100,
    'min': analysis[model_cols].min(),
    'median': analysis[model_cols].median(),
    'max': analysis[model_cols].max()
})
display(quality)

model_data = analysis.dropna(subset=['price_growth_pct', 'lag_population_growth_pct', 'lag_ntn_growth_pct', 'region', 'year']).copy()
print(f'Model observations: {len(model_data):,}')
print(f'Municipalities: {model_data["municipality_code"].nunique():,}')
print(f'Years: {model_data["year"].min()}–{model_data["year"].max()}')
display(model_data[model_cols].describe().T)

## 5. Pearson and Spearman correlations

Pearson measures linear association; Spearman measures monotonic association based on ranks. Agreement between the two provides more confidence that the result is not driven only by scale or extreme observations. Neither coefficient establishes causality.

In [ ]:
corr_cols = ['price_growth_pct', 'lag_population_growth_pct', 'lag_ntn_growth_pct', 'lag_price_growth_pct']
pearson = model_data[corr_cols].corr(method='pearson')
spearman = model_data[corr_cols].corr(method='spearman')
print('Pearson correlation')
display(pearson)
print('Spearman correlation')
display(spearman)

## 6. Bivariate relationships

Scatter plots make the statistical relationship visible before introducing controls. The fitted line is descriptive; it should not be interpreted as a causal response function.

In [ ]:
for x, label in [('lag_population_growth_pct', 'Lagged population growth (%)'), ('lag_ntn_growth_pct', 'Lagged NTN growth (%)')]:
    fig, ax = plt.subplots(figsize=(9, 6))
    sample = model_data[[x, 'price_growth_pct']].dropna()
    ax.scatter(sample[x], sample['price_growth_pct'], alpha=0.25)
    if len(sample) > 1:
        z = np.polyfit(sample[x], sample['price_growth_pct'], 1)
        x_grid = np.linspace(sample[x].quantile(0.01), sample[x].quantile(0.99), 100)
        ax.plot(x_grid, z[0] * x_grid + z[1], linewidth=2)
    ax.set_title(f'Price growth vs {label}')
    ax.set_xlabel(label)
    ax.set_ylabel('Current price growth (%)')
    fig.tight_layout()
    plt.show()

## 7. OLS regression with region and year effects

The baseline model estimates the association between current price growth and lagged population/transaction growth while controlling for broad regional and time effects. HC3 heteroskedasticity-robust standard errors are used for inference.

In [ ]:
formula = ('price_growth_pct ~ lag_population_growth_pct + lag_ntn_growth_pct + '
           'C(region) + C(year)')
ols = smf.ols(formula, data=model_data).fit(cov_type='HC3')
print(ols.summary())

### Coefficient interpretation

For example, β for `lag_ntn_growth_pct` is the estimated change in current annual price growth associated with a one-percentage-point increase in lagged NTN growth, conditional on the other variables in the specification. Statistical significance does not establish economic importance or causality.

In [ ]:
coef_table = pd.DataFrame({
    'coefficient': ols.params,
    'std_error_HC3': ols.bse,
    'p_value': ols.pvalues,
    'ci_low': ols.conf_int()[0],
    'ci_high': ols.conf_int()[1]
})
display(coef_table.loc[[c for c in coef_table.index if 'lag_' in c]])
print(f'R²: {ols.rsquared:.4f}')
print(f'Adjusted R²: {ols.rsquared_adj:.4f}')

## 8. Persistence model

Price growth can exhibit temporal persistence. Adding lagged price growth tests whether demographic and transaction variables retain an association after accounting for the municipality's previous price dynamics.

In [ ]:
dynamic_data = model_data.dropna(subset=['lag_price_growth_pct']).copy()
dynamic_formula = ('price_growth_pct ~ lag_population_growth_pct + lag_ntn_growth_pct + '
                   'lag_price_growth_pct + C(region) + C(year)')
dynamic_ols = smf.ols(dynamic_formula, data=dynamic_data).fit(cov_type='HC3')
display(pd.DataFrame({
    'coefficient': dynamic_ols.params,
    'std_error_HC3': dynamic_ols.bse,
    'p_value': dynamic_ols.pvalues
}).loc[[c for c in dynamic_ols.params.index if 'lag_' in c]])
print(f'Baseline R²: {ols.rsquared:.4f}')
print(f'Dynamic R²: {dynamic_ols.rsquared:.4f}')

## 9. Residual diagnostics

OLS residual diagnostics are used to identify obvious model problems. Heteroskedasticity is tested with Breusch–Pagan; robust HC3 inference already reduces sensitivity of standard errors to heteroskedasticity.

In [ ]:
bp = het_breuschpagan(ols.resid, ols.model.exog)
bp_table = pd.Series({'LM statistic': bp[0], 'LM p-value': bp[1], 'F statistic': bp[2], 'F p-value': bp[3]}).to_frame('value')
display(bp_table)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(ols.fittedvalues, ols.resid, alpha=0.25)
ax.axhline(0, linestyle='--', linewidth=1)
ax.set_title('OLS residuals vs fitted values')
ax.set_xlabel('Fitted price growth (%)')
ax.set_ylabel('Residual')
fig.tight_layout()
plt.show()

## 10. Robustness check: trim extreme growth observations

Real-estate growth rates can contain extreme values for small municipalities or low-volume markets. We therefore compare the baseline model with a 1st–99th percentile trim on the three growth variables used in the specification.

In [ ]:
trim_cols = ['price_growth_pct', 'lag_population_growth_pct', 'lag_ntn_growth_pct']
robust = model_data.copy()
for col in trim_cols:
    lo, hi = robust[col].quantile([0.01, 0.99])
    robust = robust[robust[col].between(lo, hi)]

robust_ols = smf.ols(formula, data=robust).fit(cov_type='HC3')
comparison = pd.DataFrame({
    'baseline_coef': ols.params,
    'baseline_pvalue': ols.pvalues,
    'trimmed_coef': robust_ols.params.reindex(ols.params.index),
    'trimmed_pvalue': robust_ols.pvalues.reindex(ols.params.index)
})
display(comparison.loc[[c for c in comparison.index if 'lag_' in c]])
print(f'Baseline observations: {len(model_data):,}')
print(f'Trimmed observations: {len(robust):,}')

## 11. Analytical interpretation framework

The correct reading of this notebook is:

1. **Descriptive evidence:** correlations show whether variables move together.
2. **Conditional association:** OLS estimates the relationship after controlling for region and year effects.
3. **Robustness:** similar coefficients after trimming extremes are more reassuring than a result driven by a few observations.
4. **No causal claim:** the design does not identify causal effects because omitted variables, reverse causality and measurement differences remain possible.
5. **Economic significance:** even a statistically significant coefficient should be assessed for its magnitude and practical relevance.

Potential omitted factors include mortgage rates, income, employment, housing supply, tourism, migration, local construction activity and macroeconomic conditions.

## 12. Key limitations

- **OMI quotations are not transaction prices:** they represent market quotation ranges and are aggregated here through their midpoint.
- **NTN is transaction volume:** it does not measure the price paid or property characteristics.
- **Population is annual:** it is repeated across semesters in the source panel and therefore must not be interpreted as a six-month observation.
- **Municipality heterogeneity:** small municipalities can show unstable percentage changes.
- **Selection and coverage:** municipalities may have missing OMI, NTN or population observations.
- **Endogeneity:** price and transaction activity can influence each other. Lagging variables reduces simultaneity concerns but does not solve endogeneity.
- **Panel dependence:** observations for the same municipality across years are not fully independent; future work should consider municipality-clustered standard errors or dedicated panel estimators.

> The next methodological step is therefore not simply a more complex regression: it is a properly specified panel model and, if the data permit, a forecasting exercise with strict temporal train/test separation.

## 13. Conclusion and hand-off to the next notebook

This notebook upgrades the project from **market description and segmentation** to **hypothesis-driven statistical analysis**. The next step can build a dedicated forecasting notebook using lagged market indicators, rolling features and time-based validation.

A strong portfolio narrative is now: **data quality → descriptive market structure → integrated market panel → segmentation → statistical association → forecasting**.